#### Feature Selection — Model HỆ SỐ NHÂN (bộ TP.HCM)

Mục tiêu: chọn bộ feature tốt nhất để dự đoán `target_shown_multiplier`.

Hệ số nhân do cung–cầu quyết định trực tiếp; giờ/vị trí/thời tiết tác động gián tiếp qua cung–cầu.

Notebook chạy **5 phương pháp** chọn feature, mỗi phương pháp trình bày 4 mục: *phương pháp là gì ·
cách thực hiện · thực hiện · bảng feature chọn được*. Cuối cùng **đối chiếu** kết quả 5 phương pháp
để ra bộ feature chốt.

Năm phương pháp: (1) Correlation & η · (2) Mutual Information · (3) Permutation importance ·
(4) RFE / backward selection · (5) SHAP.

**Chuẩn bị: nạp dữ liệu, feature ứng viên, mã hoá cho RFE/SHAP**

In [1]:
import sys; sys.path.insert(0, ".")
from _common import *
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_regression, RFE
from sklearn.preprocessing import OrdinalEncoder
setup()
df = load(frac=0.12)   # doi frac=1.0 khi chay that

CAT = ["service_name", "pickup_location_name", "dropoff_location_name", "weather_main"]
NUM = ["pricing_demand_index_5m_lag", "pricing_supply_index_5m_lag", "pricing_market_imbalance_5m_lag", "pricing_quote_count_5m_lag", "pricing_avg_shown_multiplier_5m_lag", "gio_vn", "thu_vn", "target_is_weekend", "latest_observed_multiplier", "history_60m_multiplier_mean", "requested_lag_minutes", "actual_observation_age_minutes", "weather_temp", "weather_humidity", "weather_clouds_all"]
FEATS = CAT + NUM
TARGET = "target_shown_multiplier"
K = 12   # so feature chon o moi phuong phap

# Chong leakage
CAM = {"target_shown_price","target_shown_multiplier","target_price_per_km",
       "split","scenario_id","forecast_example_id","target_request_id","is_synthetic"}
assert not (set(FEATS) & CAM), "CO FEATURE LEAKAGE!"
print(f"Model HỆ SỐ NHÂN: {len(FEATS)} feature ung vien ({len(CAT)} cat + {len(NUM)} num) | chon top K={K}")

name_vn = "HE SO NHAN"
def prep(d, cat):
    X=d.copy()
    for c in cat: X[c]=X[c].astype("category")
    return X

# Ma hoa so cho RFE/SHAP (RandomForest can so + khong NaN)
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
Xrf = df[FEATS].copy()
Xrf[CAT] = enc.fit_transform(Xrf[CAT].astype(str))
Xrf = Xrf.fillna(Xrf.median(numeric_only=True))
y = df[TARGET].values

SELECTED = {}   # luu bo feature chon cua tung phuong phap
def bang_chon(rank_series, ten):
    sel = list(rank_series.head(K).index)
    SELECTED[ten] = sel
    print(f"[{ten}] chon {len(sel)} feature:"); print("   " + ", ".join(sel))
    return rank_series.to_frame("diem")
print("San sang.")

Nap 827,693 dong (mau 12%) | gia median 114k VND | surge 81.7%
Model HỆ SỐ NHÂN: 19 feature ung vien (4 cat + 15 num) | chon top K=12
San sang.


#### 1. Correlation và Correlation ratio (η)

**Phương pháp là gì:** đo quan hệ trực tiếp giữa từng feature với target, không cần train model
(nhóm *filter*). Nhanh, đơn giản.

**Cách thực hiện:** biến số dùng |Pearson correlation|; biến phân loại dùng η (0–1, % phương sai
target giải thích bởi nhóm). Xếp hạng, chọn top K.

In [2]:
# Thuc hien
rows={}
for f in CAT: rows[f]=eta(df[f], df[TARGET])
for f in NUM: rows[f]=abs(df[f].corr(df[TARGET]))
rank_corr = pd.Series(rows).sort_values(ascending=False)
# Ket qua
print("Ket qua — xep hang + feature chon:")
display(bang_chon(rank_corr, "Correlation/eta").round(4))

Ket qua — xep hang + feature chon:
[Correlation/eta] chon 12 feature:
   latest_observed_multiplier, pricing_avg_shown_multiplier_5m_lag, history_60m_multiplier_mean, pricing_market_imbalance_5m_lag, pricing_demand_index_5m_lag, gio_vn, pricing_quote_count_5m_lag, pickup_location_name, pricing_supply_index_5m_lag, weather_temp, weather_humidity, target_is_weekend


,diem
latest_observed_multiplier,0.9509
pricing_avg_shown_multiplier_5m_lag,0.9087
history_60m_multiplier_mean,0.8498
pricing_market_imbalance_5m_lag,0.7979
pricing_demand_index_5m_lag,0.6885
gio_vn,0.4738
pricing_quote_count_5m_lag,0.4478
pickup_location_name,0.2988
pricing_supply_index_5m_lag,0.2759
weather_temp,0.2066


#### 2. Mutual Information

**Phương pháp là gì:** đo lượng thông tin feature cung cấp về target — bắt được cả quan hệ **phi
tuyến** mà correlation bỏ sót (nhóm *filter*).

**Cách thực hiện:** tính MI của từng feature (đã mã hoá số) với target trên một mẫu; xếp hạng, chọn top K.

In [3]:
# Thuc hien
samp = np.random.RandomState(0).choice(len(Xrf), min(40000,len(Xrf)), replace=False)
mi = mutual_info_regression(Xrf.iloc[samp], y[samp], random_state=0)
rank_mi = pd.Series(mi, index=FEATS).sort_values(ascending=False)
# Ket qua
print("Ket qua — xep hang + feature chon:")
display(bang_chon(rank_mi, "MutualInfo").round(4))

Ket qua — xep hang + feature chon:
[MutualInfo] chon 12 feature:
   latest_observed_multiplier, pricing_market_imbalance_5m_lag, pricing_avg_shown_multiplier_5m_lag, history_60m_multiplier_mean, pricing_demand_index_5m_lag, weather_humidity, gio_vn, weather_clouds_all, weather_temp, pricing_supply_index_5m_lag, pricing_quote_count_5m_lag, pickup_location_name


,diem
latest_observed_multiplier,1.2544
pricing_market_imbalance_5m_lag,1.1454
pricing_avg_shown_multiplier_5m_lag,0.9570
history_60m_multiplier_mean,0.6636
pricing_demand_index_5m_lag,0.6303
weather_humidity,0.4644
gio_vn,0.4177
weather_clouds_all,0.3979
weather_temp,0.3709
pricing_supply_index_5m_lag,0.2705


#### 3. Permutation importance

**Phương pháp là gì:** đo feature mà **model thực sự dùng** — xáo trộn ngẫu nhiên 1 feature rồi
xem sai số tăng bao nhiêu (nhóm *embedded*). Đáng tin vì tính cả tương tác.

**Cách thực hiện:** train HistGradientBoosting trên tập train; trên tập validation, xáo trộn từng
feature, đo mức tăng sai số; xếp hạng, chọn top K.

In [4]:
# Thuc hien
tr = df[df.split=="train"].sample(min(60000,(df.split=="train").sum()), random_state=1)
va = df[df.split=="validation"].sample(min(20000,(df.split=="validation").sum()), random_state=2)
m = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.08, categorical_features=CAT,
    random_state=42).fit(prep(tr,CAT)[FEATS], tr[TARGET])
pi = permutation_importance(m, prep(va,CAT)[FEATS], va[TARGET], n_repeats=3, random_state=0, n_jobs=1)
rank_perm = pd.Series(pi.importances_mean, index=FEATS).sort_values(ascending=False)
# Ket qua
print("Ket qua — xep hang + feature chon:")
display(bang_chon(rank_perm, "Permutation").round(4))

Ket qua — xep hang + feature chon:
[Permutation] chon 12 feature:
   latest_observed_multiplier, pricing_market_imbalance_5m_lag, pricing_avg_shown_multiplier_5m_lag, gio_vn, history_60m_multiplier_mean, thu_vn, target_is_weekend, pickup_location_name, actual_observation_age_minutes, weather_main, pricing_supply_index_5m_lag, pricing_demand_index_5m_lag


,diem
latest_observed_multiplier,0.6712
pricing_market_imbalance_5m_lag,0.2481
pricing_avg_shown_multiplier_5m_lag,0.1358
gio_vn,0.0587
history_60m_multiplier_mean,0.0355
thu_vn,0.0204
target_is_weekend,0.0154
pickup_location_name,0.0088
actual_observation_age_minutes,0.0051
weather_main,0.0038


#### 4. RFE — Recursive Feature Elimination (backward selection)

**Phương pháp là gì:** chọn feature theo kiểu *wrapper* — bắt đầu với **tất cả** feature, **loại
dần** feature yếu nhất, train lại, lặp đến khi còn K feature. Đây chính là *backward selection*.

**Cách thực hiện:** dùng RandomForest (có `feature_importances_`) làm bộ đánh giá; RFE loại 2
feature mỗi vòng cho tới khi còn K. Feature còn lại là bộ được chọn.

In [5]:
# Thuc hien (RandomForest can so -> dung Xrf da ma hoa)
s = np.random.RandomState(3).choice(len(Xrf), min(25000,len(Xrf)), replace=False)
rf = RandomForestRegressor(n_estimators=60, max_depth=14, n_jobs=-1, random_state=42)
rfe = RFE(rf, n_features_to_select=K, step=2).fit(Xrf.iloc[s], y[s])
rank_rfe = pd.Series(rfe.ranking_, index=FEATS).sort_values()   # 1 = duoc chon
SELECTED["RFE"] = [f for f,keep in zip(FEATS, rfe.support_) if keep]
# Ket qua
print(f"[RFE] chon {len(SELECTED['RFE'])} feature:"); print("   " + ", ".join(SELECTED["RFE"]))
print("\n(ranking: 1 = duoc chon, so cang lon = bi loai cang som)")
display(rank_rfe.to_frame("ranking"))

[RFE] chon 12 feature:
   weather_main, pricing_demand_index_5m_lag, pricing_supply_index_5m_lag, pricing_market_imbalance_5m_lag, pricing_avg_shown_multiplier_5m_lag, gio_vn, thu_vn, latest_observed_multiplier, history_60m_multiplier_mean, actual_observation_age_minutes, weather_temp, weather_clouds_all

(ranking: 1 = duoc chon, so cang lon = bi loai cang som)


,ranking
weather_main,1
pricing_market_imbalance_5m_lag,1
pricing_supply_index_5m_lag,1
pricing_demand_index_5m_lag,1
latest_observed_multiplier,1
thu_vn,1
gio_vn,1
pricing_avg_shown_multiplier_5m_lag,1
actual_observation_age_minutes,1
history_60m_multiplier_mean,1


#### 5. SHAP

**Phương pháp là gì:** đo đóng góp của từng feature vào **từng dự đoán** (dựa trên lý thuyết trò
chơi Shapley), rồi lấy trung bình |đóng góp| làm mức quan trọng. Diễn giải sâu, có chiều (nhóm *embedded*).

**Cách thực hiện:** train RandomForest, dùng `TreeExplainer` tính SHAP values trên một mẫu nhỏ;
lấy mean(|SHAP|) mỗi feature; xếp hạng, chọn top K.

In [6]:
# Thuc hien
try:
    import shap
    s2 = np.random.RandomState(5).choice(len(Xrf), min(20000,len(Xrf)), replace=False)
    rf2 = RandomForestRegressor(n_estimators=60, max_depth=14, n_jobs=-1, random_state=42).fit(Xrf.iloc[s2], y[s2])
    Xs = Xrf.iloc[np.random.RandomState(6).choice(len(Xrf), 1500, replace=False)]
    sv = shap.TreeExplainer(rf2).shap_values(Xs)
    rank_shap = pd.Series(np.abs(sv).mean(0), index=FEATS).sort_values(ascending=False)
    print("Ket qua — xep hang + feature chon:")
    display(bang_chon(rank_shap, "SHAP").round(4))
except ImportError:
    print("Chua cai shap. Chay: pip install shap")
    SELECTED["SHAP"] = []

Ket qua — xep hang + feature chon:
[SHAP] chon 12 feature:
   latest_observed_multiplier, pricing_market_imbalance_5m_lag, history_60m_multiplier_mean, pricing_avg_shown_multiplier_5m_lag, weather_main, gio_vn, thu_vn, actual_observation_age_minutes, target_is_weekend, pricing_demand_index_5m_lag, weather_clouds_all, pricing_supply_index_5m_lag


,diem
latest_observed_multiplier,0.0979
pricing_market_imbalance_5m_lag,0.0467
history_60m_multiplier_mean,0.0049
pricing_avg_shown_multiplier_5m_lag,0.0049
weather_main,0.0045
gio_vn,0.0033
thu_vn,0.0023
actual_observation_age_minutes,0.0023
target_is_weekend,0.0018
pricing_demand_index_5m_lag,0.0015


#### Kiểm tra trùng lặp (bổ trợ)

Hai feature |r| > 0,9 mang thông tin trùng → nên bỏ bớt 1 (model gọn, importance không chia phiếu).

In [7]:
import itertools
corr = df[NUM].corr().abs()
trung=[(a,b,round(corr.loc[a,b],3)) for a,b in itertools.combinations(NUM,2) if corr.loc[a,b]>0.9]
if trung:
    print("Cap feature TRUNG LAP (|r|>0.9) -> can nhac bo bot 1:")
    for a,b,r in sorted(trung,key=lambda x:-x[2]): print(f"   {a:36} <-> {b:36} r={r}")
else:
    print("Khong co cap |r|>0.9.")

Cap feature TRUNG LAP (|r|>0.9) -> can nhac bo bot 1:
   pricing_avg_shown_multiplier_5m_lag  <-> history_60m_multiplier_mean          r=0.983
   requested_lag_minutes                <-> actual_observation_age_minutes       r=0.972
   pricing_avg_shown_multiplier_5m_lag  <-> latest_observed_multiplier           r=0.962
   latest_observed_multiplier           <-> history_60m_multiplier_mean          r=0.926
   weather_temp                         <-> weather_humidity                     r=0.924


#### Đối chiếu 5 phương pháp và chốt bộ feature

Đếm mỗi feature được **bao nhiêu phương pháp chọn** (bỏ phiếu). Feature được **nhiều phương pháp
đồng thuận** thì càng đáng giữ. Bộ chốt = feature được ≥ 3/5 phương pháp chọn (kết hợp loại trùng lặp).

In [8]:
# Bang doi chieu: moi feature x moi phuong phap (1 = duoc chon)
pps = ["Correlation/eta","MutualInfo","Permutation","RFE","SHAP"]
dc = pd.DataFrame(0, index=FEATS, columns=pps)
for pp in pps:
    for f in SELECTED.get(pp,[]): dc.loc[f,pp]=1
dc["so_pp_chon"] = dc.sum(axis=1)
dc = dc.sort_values("so_pp_chon", ascending=False)
print("BANG DOI CHIEU 5 PHUONG PHAP (1 = duoc chon):")
display(dc)

BANG DOI CHIEU 5 PHUONG PHAP (1 = duoc chon):


,Correlation/eta,MutualInfo,Permutation,RFE,SHAP,so_pp_chon
pricing_market_imbalance_5m_lag,1,1,1,1,1,5
pricing_supply_index_5m_lag,1,1,1,1,1,5
pricing_demand_index_5m_lag,1,1,1,1,1,5
gio_vn,1,1,1,1,1,5
latest_observed_multiplier,1,1,1,1,1,5
history_60m_multiplier_mean,1,1,1,1,1,5
pricing_avg_shown_multiplier_5m_lag,1,1,1,1,1,5
weather_clouds_all,0,1,0,1,1,3
thu_vn,0,0,1,1,1,3
pickup_location_name,1,1,1,0,0,3


In [9]:
# Bo feature CHOT: duoc >= 3/5 phuong phap chon
chot = list(dc[dc.so_pp_chon >= 3].index)
chot_cat = [f for f in chot if f in CAT]
chot_num = [f for f in chot if f in NUM]
print("="*60)
print(f"BO FEATURE CHOT — MODEL {name_vn} (>= 3/5 phuong phap dong thuan)")
print("="*60)
print(f"  Categorical ({len(chot_cat)}): {chot_cat}")
print(f"  Numeric     ({len(chot_num)}): {chot_num}")
print(f"  Tong: {len(chot)} feature  |  Target: {TARGET}")
print("\n  (Co the chinh tay: bo bot cap trung lap o buoc tren neu con)")

BO FEATURE CHOT — MODEL HE SO NHAN (>= 3/5 phuong phap dong thuan)
  Categorical (2): ['pickup_location_name', 'weather_main']
  Numeric     (12): ['pricing_market_imbalance_5m_lag', 'pricing_supply_index_5m_lag', 'pricing_demand_index_5m_lag', 'gio_vn', 'latest_observed_multiplier', 'history_60m_multiplier_mean', 'pricing_avg_shown_multiplier_5m_lag', 'weather_clouds_all', 'thu_vn', 'weather_temp', 'actual_observation_age_minutes', 'target_is_weekend']
  Tong: 14 feature  |  Target: target_shown_multiplier

  (Co the chinh tay: bo bot cap trung lap o buoc tren neu con)
